# Aegis — End-to-End Demo (mocked)

**Purpose:** Demonstrates a full mission run for *Aegis: Self-Evolving Agent Fleet*.
This notebook is fully runnable without API keys (mocked agents & judge).
Replace the mocked parts with real LLM calls in production.

**Contents**
- Mission submission
- Planner & orchestrator
- Specialist agents (registered in MCP-style registry)
- Trace instrumentation (simple spans)
- LLM-as-Judge (mocked)
- AgentCreator generates a new agent when judge score is low
- Re-run showing improved results


In [ ]:
# Install optional helper libs
!pip install rich --quiet

In [ ]:
import time, uuid, json
from rich.pretty import pprint

# Global trace store
TRACES = []

def clear_traces():
    global TRACES
    TRACES = []

In [ ]:
def trace_start(span_name, meta=None):
    span = {
        "id": str(uuid.uuid4()),
        "name": span_name,
        "start": time.time(),
        "meta": meta or {},
        "events": []
    }
    TRACES.append(span)
    return span

def trace_end(span, result):
    span["end"] = time.time()
    span["duration"] = span["end"] - span["start"]
    span["result"] = result
    span["events"].append({"event":"end","t":time.time()})

In [ ]:
AGENT_REGISTRY = {}

def register_agent(name, description, func):
    AGENT_REGISTRY[name] = {"name": name, "description": description, "callable": func}
    return AGENT_REGISTRY[name]

def call_agent(name, args, context):
    if name not in AGENT_REGISTRY:
        raise ValueError(f"Agent not found: {name}")
    span = trace_start(f"agent_call:{name}", {"args": args})
    res = AGENT_REGISTRY[name]["callable"](args, context)
    trace_end(span, res)
    return res

In [ ]:
def market_research_agent(args, ctx):
    q = args.get("query")
    # Mocked research output
    return {"insights": f"Mocked market insights for '{q}'", "confidence": 0.85}

def copy_agent(args, ctx):
    brief = args.get("brief")
    return {"copy": f"Mock headline for '{brief}': Aegis — launch faster.", "confidence": 0.92}

def webdev_agent(args, ctx):
    # Mock creation of an artifact
    return {"artifact": {"url": "https://example.local/landing.html"}, "confidence": 0.9}

# Register
register_agent("MarketResearchAgent", "Gathers market insights", market_research_agent)
register_agent("CopyAgent", "Writes marketing copy", copy_agent)
register_agent("WebDevAgent", "Deploys landing pages", webdev_agent)

In [ ]:
def planner_create_plan(mission_text):
    # Very simple plan decomposition (mock)
    return [
        {"step": "research", "agent": "MarketResearchAgent", "args": {"query": mission_text}},
        {"step": "copywriting", "agent": "CopyAgent", "args": {"brief": mission_text}},
        {"step": "deploy", "agent": "WebDevAgent", "args": {"brief": mission_text}}
    ]

def orchestrate_mission(mission_text, context=None):
    if context is None: context = {}
    clear_traces()
    plan = planner_create_plan(mission_text)
    results = {}
    for s in plan:
        res = call_agent(s["agent"], s["args"], context)
        results[s["step"]] = res
    return {"plan": plan, "results": results, "trace": TRACES}

In [ ]:
def llm_judge(mission, trace, golden=None):
    # Mocked scoring that rewards presence of specific agents
    called_agents = [span["name"] for span in trace]
    score = 0.0
    if any("MarketResearchAgent" in n for n in called_agents): score += 0.35
    if any("CopyAgent" in n for n in called_agents): score += 0.35
    if any("WebDevAgent" in n for n in called_agents): score += 0.25
    # Cap at 1.0
    score = round(min(score, 1.0), 2)
    return {"score": score, "notes": "Mock judge: higher score when core agents used."}

In [ ]:
def agent_creator_suggest(missing_capability):
    # Create a minimal agent spec
    name = f"{missing_capability}Agent"
    spec = {
        "name": name,
        "description": f"Auto-generated agent to handle {missing_capability}",
        "example_input": {"task": "analyze X"},
        "example_output": {"report": "Y"}
    }
    return spec

# Example implementation for a generated AnalyticsAgent
def analytics_agent(args, ctx):
    return {"report": f"Mock analytics for {args.get('campaign','unknown')}", "confidence": 0.95}

## Run initial mission
We'll run a mission and show the plan, intermediate agent outputs, trace, and judge score.

In [ ]:
mission = "Launch a micro marketing campaign for Product X targeting students in Bangalore"
out = orchestrate_mission(mission)
print("Plan:")
pprint(out["plan"])
print("\nResults:")
pprint(out["results"])
print("\nTrace spans (chronological):")
pprint(out["trace"])
judge = llm_judge(mission, out["trace"])
print("\nJudge score:")
pprint(judge)

## Auto-evolution step
If the judge score is below threshold (say 0.9), AgentCreator will generate a new agent spec and register it.
Then we re-run the mission with the newly added agent included to demonstrate improvement.

In [ ]:
THRESHOLD = 0.9
if out and judge["score"] < THRESHOLD:
    missing = "Analytics"  # mock detection logic
    spec = agent_creator_suggest(missing)
    print("AgentCreator suggested spec:")
    pprint(spec)
    # Register actual implementation (mock)
    register_agent(spec["name"], spec["description"], analytics_agent)
    print(f"Registered new agent: {spec['name']}")
    # Re-run mission but include analytics step at the end of plan (demo override)
    def orchestrate_with_analytics(mission_text, context=None):
        if context is None: context = {}
        clear_traces()
        plan = planner_create_plan(mission_text) + [{"step":"analytics","agent":spec["name"], "args":{"campaign":"Product X"}}]
        results = {}
        for s in plan:
            res = call_agent(s["agent"], s["args"], context)
            results[s["step"]] = res
        return {"plan": plan, "results": results, "trace": TRACES}

    out2 = orchestrate_with_analytics(mission)
    print("\nRe-run plan:")
    pprint(out2["plan"])
    print("\nRe-run results:")
    pprint(out2["results"])
    print("\nRe-run trace:")
    pprint(out2["trace"])
    judge2 = llm_judge(mission, out2["trace"])
    print("\nNew judge score:")
    pprint(judge2)
else:
    print("Judge score high enough; no auto-evolution triggered.")

## Conclusion

This demo shows:
- Planner decomposes mission into steps
- Root orchestrator invokes specialist agents via a registry
- Each agent emits a trace span for observability
- LLM-as-Judge evaluates the entire trajectory
- AgentCreator can generate and register a new agent, improving future runs

**Next steps to make this production-grade:**
- Replace mocked agents & judge with Gemini or another LLM (careful with API keys)
- Add persistent session & long-term memory (vector DB)
- Add OpenTelemetry tracing and a Grafana dashboard
- Integrate MCP gateway for tool schema enforcement